In [5]:
import numpy as np
import pymc as pm

R = 2.0
N = 40
r = 0.1
a = 0.5
Y = 10000
energy = 1.0  # 1 МэВ

pmt_angular_radius = np.arcsin(r / R)
omega_pmt = 2 * np.pi * (1.0 - np.cos(pmt_angular_radius))
total_sphere_solid_angle = 4 * np.pi
geometric_efficiency = (N * omega_pmt) / total_sphere_solid_angle

mean_distance = 0.75 * R
attenuation_length = 8.0
survival_prob = np.exp(-mean_distance / attenuation_length)

effective_p = geometric_efficiency * survival_prob

with pm.Model() as detector_model:
    # число испущенных фотонов 
    n_photons = pm.Poisson('n_photons', mu=energy * Y)
    
    # число фотонов, попавших на все ФЭУ 
    n_hits = pm.Binomial('n_hits', n=n_photons, p=effective_p)
    
    # полный собранный заряд 
    total_charge = pm.Poisson('total_charge', mu=a * n_hits)
    
    prior_samples = pm.sample_prior_predictive(draws=1000, random_seed=42)

pymc_charges = prior_samples.prior['total_charge'].values.flatten()

mean_pymc = np.mean(pymc_charges)
std_pymc = np.std(pymc_charges)
resolution_pymc = std_pymc / mean_pymc

print(f"Средний заряд: {mean_pymc:.1f} ф.э.")
print(f"СКО (σ): {std_pymc:.1f}")
print(f"Разрешение (σ/mean): {resolution_pymc:.4f}")

Sampling: [n_hits, n_photons, total_charge]


Средний заряд: 103.5 ф.э.
СКО (σ): 12.9
Разрешение (σ/mean): 0.1249
